# SVC



## Creamos el pipeline del modelo

El dataset está mezclado en train, val y test ya que val lo vamos a usar en deep learning para hacer early stopping. Pero en shallow learning para comparar modelos con resultados más representativos del rendimiento conviene más usar cross validation. 

Para usar cross validation, tenemos que juntar train+val. 

Aquí surge un problema, y es que anteriormente en el notebook 2 hicimos fit transform del vectorizer en train y transform en val lo cual para usarlos de forma separada está bien y no causa problemas pero al juntarlos para hacer cross validation es posible que en algún fold haya un validation set de parte del train con el que se haya hecho fit, metiendo data leakage. 

Para ello usaremos un pipeline y volveremos a vectorizar desde los tokens. 

## TF-iDF


In [6]:
import pandas as pd
from jailbreak_detection.features.tfidf import build_vectorizer, flatten

train = pd.read_json("../data/interim/train_tok.jsonl", lines=True)
X_train = train["tokens"].map(flatten)   # listas de tokens por prompt
y_train = train["jailbreak"]


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

# 1. Definir los modelos
models = {
    "LinearSVC": Pipeline([
        ("vec", build_vectorizer(min_df=2)),
        ("clf", LinearSVC(class_weight="balanced"))
    ])
}

# 2. Cross-Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Evaluar todos los modelos
results = []

for name, pipe in models.items():

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring = ["f1", "accuracy","precision","recall","matthews_corrcoef"],
        n_jobs=-1
    )

    results.append({
        "Modelo": name,
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Accuracy": scores["test_accuracy"].mean(),
        "Accuracy_std": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Precision_std": scores["test_precision"].std(),
        "Recall": scores["test_recall"].mean(),
        "Recall_std": scores["test_recall"].std(),
        "MCC": scores["test_matthews_corrcoef"].mean(),
        "MCC_std": scores["test_matthews_corrcoef"].std(),

    })


# 4. Comparar resultados
df_results = pd.DataFrame(results)
print(df_results.sort_values("F1", ascending=False))

# Guardamos una copia para la tabla comparativa final (la celda de BoW reutiliza df_results)
df_results_tfidf = df_results.copy()

 resutlaods optuna: 
 Mejor MCC: 0.6128893579281603

 Mejores parámetros: {'C': 1.9892802274290187, 'min_df': 1}

In [12]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
import json

# Definimos los folds una sola vez
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    # Hiperparámetros propuestos por Optuna
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    min_df = trial.suggest_int("min_df", 1, 5)

    # Nuevos hiperparámetros del vectorizador
    max_ngram = trial.suggest_categorical(
        "max_ngram", [1, 2]
    )

    # Nuevos hiperparámetros de LinearSVC
    penalty = trial.suggest_categorical(
        "penalty", ["l1", "l2"]
    )

    class_weight = trial.suggest_categorical(
        "class_weight", [None, "balanced"]
    )

    # Evitar combinaciones incompatibles
    if penalty == "l1":
        loss = "squared_hinge"
        dual = False
    else:
        loss = trial.suggest_categorical(
            "loss", ["hinge", "squared_hinge"]
        )
        dual = True if loss == "hinge" else "auto"

    pipe = Pipeline([
        ("vec", build_vectorizer(min_df=min_df).set_params(
            ngram_range=(1, max_ngram)
        )),
        ("clf", LinearSVC(
            C=C,
            penalty=penalty,
            loss=loss,
            dual=dual,
            class_weight=class_weight,
            random_state=42
        ))
    ])

    # Cross-Validation automática
    scores = cross_val_score(
        estimator=pipe,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="matthews_corrcoef",
        n_jobs=-1
    )

    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Mejor Mcc:", study.best_value)
print("Mejores parámetros:", study.best_params)

#guardamos los best params para no tener que runnear optuna cada vez. 
with open("../data/processed/svc_tfidf_best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

[I 2026-11-11 09:13:41,746] A new study created in memory with name: no-name-5f8b20f5-b435-4ae3-8ef7-f939206ec360
[I 2026-11-11 09:14:09,957] Trial 0 finished with value: 0.6206883957429448 and parameters: {'C': 1.224668457606394, 'min_df': 1, 'max_ngram': 2, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.6206883957429448.
[I 2026-11-11 09:14:26,670] Trial 1 finished with value: 0.0 and parameters: {'C': 0.005199876080444764, 'min_df': 5, 'max_ngram': 2, 'penalty': 'l1', 'class_weight': None}. Best is trial 0 with value: 0.6206883957429448.
[I 2026-11-11 09:14:42,618] Trial 2 finished with value: 0.6107429681754428 and parameters: {'C': 3.4422300149191223, 'min_df': 5, 'max_ngram': 2, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'squared_hinge'}. Best is trial 0 with value: 0.6206883957429448.
[I 2026-11-11 09:14:50,074] Trial 3 finished with value: 0.6128893579281603 and parameters: {'C': 1.9755142800397028, 'min_df': 1, 'm

Mejor Mcc: 0.6229252922780387
Mejores parámetros: {'C': 0.7720899698248374, 'min_df': 2, 'max_ngram': 2, 'penalty': 'l2', 'class_weight': 'balanced', 'loss': 'squared_hinge'}


In [13]:
# ---------- Evaluación final en test (TF-IDF) con los mejores hiperparámetros de Optuna ----------
from sklearn.metrics import classification_report, matthews_corrcoef, f1_score


test = pd.read_json("../data/interim/test_tok.jsonl", lines=True)
X_test = test["tokens"].map(flatten)
y_test = test["jailbreak"]


#cargamos los mejores params guardados en el json
with open("../data/processed/svc_tfidf_best_params.json") as f:
    best_params = json.load(f)

best_pipe = Pipeline([
    ("vec", build_vectorizer(min_df=best_params["min_df"])),
    ("clf", LinearSVC(C=best_params["C"], class_weight="balanced", random_state=42)),
])




# El vectorizador se ajusta solo con train; en test solo se aplica transform
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))
print("F1 :", round(f1_score(y_test, y_pred), 4))
print("MCC:", round(matthews_corrcoef(y_test, y_pred), 4))


              precision    recall  f1-score   support

       False     0.9662    0.9668    0.9665      1836
        True     0.6648    0.6612    0.6630       183

    accuracy                         0.9391      2019
   macro avg     0.8155    0.8140    0.8148      2019
weighted avg     0.9389    0.9391    0.9390      2019

F1 : 0.663
MCC: 0.6295


## BoW


In [ ]:
import pandas as pd
from jailbreak_detection.features.tfidf import flatten

train = pd.read_json("../data/interim/train_tok.jsonl", lines=True)
X_train = train["tokens"].map(flatten)   # listas de tokens por prompt
y_train = train["jailbreak"]


In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from jailbreak_detection.features.tfidf import identity


def build_bow(min_df: int = 2) -> CountVectorizer:
    """BoW sobre tokens ya calculados (mismos parámetros que en el notebook 02):
    sin volver a tokenizar, sin pasar a minúsculas. Se usa dentro del Pipeline."""
    return CountVectorizer(
        tokenizer=identity, preprocessor=identity, token_pattern=None,
        lowercase=False, min_df=min_df,
    )


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

# 1. Definir los modelos (el CountVectorizer va dentro del pipeline y parte de los tokens)
models = {
    "LinearSVC": Pipeline([
        ("vec", build_bow(min_df=2)),
        ("clf", LinearSVC(class_weight="balanced"))
    ])
}

# 2. Cross-Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Evaluar todos los modelos
results = []

for name, pipe in models.items():

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring = ["f1", "accuracy","precision","recall","matthews_corrcoef"],
        n_jobs=-1
    )

    results.append({
        "Modelo": name,
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Accuracy": scores["test_accuracy"].mean(),
        "Accuracy_std": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Precision_std": scores["test_precision"].std(),
        "Recall": scores["test_recall"].mean(),
        "Recall_std": scores["test_recall"].std(),
        "MCC": scores["test_matthews_corrcoef"].mean(),
        "MCC_std": scores["test_matthews_corrcoef"].std(),

    })


# 4. Comparar resultados
df_results = pd.DataFrame(results)
print(df_results.sort_values("F1", ascending=False))

# Guardamos una copia para la tabla comparativa final
df_results_bow = df_results.copy()

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
import json

# Definimos los folds una sola vez
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    # Hiperparámetros propuestos por Optuna
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    min_df = trial.suggest_int("min_df", 1, 5)

    # Nuevos hiperparámetros del vectorizador
    max_ngram = trial.suggest_categorical(
        "max_ngram", [1, 2]
    )

    # Nuevos hiperparámetros de LinearSVC
    penalty = trial.suggest_categorical(
        "penalty", ["l1", "l2"]
    )

    class_weight = trial.suggest_categorical(
        "class_weight", [None, "balanced"]
    )

    # Evitar combinaciones incompatibles
    if penalty == "l1":
        loss = "squared_hinge"
        dual = False
    else:
        loss = trial.suggest_categorical(
            "loss", ["hinge", "squared_hinge"]
        )
        dual = True if loss == "hinge" else "auto"

    pipe = Pipeline([
        ("vec", build_bow(min_df=min_df).set_params(
            ngram_range=(1, max_ngram)
        )),
        ("clf", LinearSVC(
            C=C,
            penalty=penalty,
            loss=loss,
            dual=dual,
            class_weight=class_weight,
            random_state=42
        ))
    ])

    # Cross-Validation automática
    scores = cross_val_score(
        estimator=pipe,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="matthews_corrcoef",
        n_jobs=-1
    )

    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Mejor Mcc:", study.best_value)
print("Mejores parámetros:", study.best_params)

#guardamos los best params para no tener que runnear optuna cada vez. 
with open("../data/processed/svc_bow_best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

In [ ]:
# ---------- Evaluación final en test (BoW) con los mejores hiperparámetros de Optuna ----------
from sklearn.metrics import classification_report, matthews_corrcoef, f1_score


test = pd.read_json("../data/interim/test_tok.jsonl", lines=True)
X_test = test["tokens"].map(flatten)
y_test = test["jailbreak"]


#cargamos los mejores params guardados en el json
with open("../data/processed/svc_bow_best_params.json") as f:
    best_params = json.load(f)

# loss y dual no siempre están en el JSON (con penalty="l1" no se sugieren): se rederivan
# con la misma regla que en objective()
if best_params["penalty"] == "l1":
    loss, dual = "squared_hinge", False
else:
    loss = best_params["loss"]
    dual = True if loss == "hinge" else "auto"

best_pipe = Pipeline([
    ("vec", build_bow(min_df=best_params["min_df"]).set_params(
        ngram_range=(1, best_params["max_ngram"])
    )),
    ("clf", LinearSVC(
        C=best_params["C"],
        penalty=best_params["penalty"],
        loss=loss,
        dual=dual,
        class_weight=best_params["class_weight"],
        random_state=42,
    )),
])


# El vectorizador se ajusta solo con train; en test solo se aplica transform
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))
print("F1 :", round(f1_score(y_test, y_pred), 4))
print("MCC:", round(matthews_corrcoef(y_test, y_pred), 4))


## Comparing models


In [11]:
# Tabla comparativa (resultados de CV con hiperparámetros por defecto, sin Optuna)
# Para añadir más filas (p. ej. los de Optuna), crea un DataFrame con las mismas columnas
# y una columna "Representación" y añádelo a la lista de abajo.
cols = ["Representación", "Modelo",
        "F1", "F1_std", "Accuracy", "Accuracy_std",
        "Precision", "Precision_std", "Recall", "Recall_std",
        "MCC", "MCC_std"]

df_compare = pd.concat(
    [
        df_results_tfidf.assign(Representación="TF-IDF"),
        df_results_bow.assign(Representación="BoW"),
    ],
    ignore_index=True,
)[cols]

df_compare = df_compare.sort_values("MCC", ascending=False).reset_index(drop=True)
df_compare.round(4)


,Representación,Modelo,F1,F1_std,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,MCC,MCC_std
0,TF-IDF,LinearSVC,0.6387,0.0275,0.9370,0.0046,0.6518,0.0302,0.6274,0.0371,0.6048,0.0297
1,BoW,Naive Bayes,0.6000,0.0348,0.9156,0.0088,0.5197,0.0371,0.7112,0.0396,0.5630,0.0387
2,BoW,LinearSVC,0.5097,0.0191,0.9128,0.0037,0.5105,0.0207,0.5110,0.0363,0.4626,0.0197
3,TF-IDF,Naive Bayes,0.0069,0.0056,0.9107,0.0008,0.4667,0.4522,0.0035,0.0028,0.0310,0.0380


### Resultados CV con Optuna (LinearSVC)


In [ ]:
import json
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate


def build_tuned_svc(make_vec, params):
    """Pipeline LinearSVC con los hiperparámetros guardados por Optuna (JSON).
    loss y dual no siempre están en el JSON (con penalty="l1" no se sugieren):
    se rederivan con la misma regla que en objective()."""
    if params["penalty"] == "l1":
        loss, dual = "squared_hinge", False
    else:
        loss = params["loss"]
        dual = True if loss == "hinge" else "auto"
    return Pipeline([
        ("vec", make_vec(min_df=params["min_df"]).set_params(
            ngram_range=(1, params["max_ngram"])
        )),
        ("clf", LinearSVC(
            C=params["C"],
            penalty=params["penalty"],
            loss=loss,
            dual=dual,
            class_weight=params["class_weight"],
            random_state=42,
        )),
    ])


# Misma CV y mismas métricas que en la tabla sin Optuna
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = {"f1": "F1", "accuracy": "Accuracy", "precision": "Precision",
           "recall": "Recall", "matthews_corrcoef": "MCC"}

optuna_configs = {
    "TF-IDF + Optuna": (build_vectorizer, "../data/processed/svc_tfidf_best_params.json"),
    "BoW + Optuna": (build_bow, "../data/processed/svc_bow_best_params.json"),
}

rows = []
for rep, (make_vec, params_path) in optuna_configs.items():
    with open(params_path) as f:
        params = json.load(f)

    scores = cross_validate(
        build_tuned_svc(make_vec, params),
        X_train, y_train,
        cv=cv, scoring=list(metrics), n_jobs=-1,
    )

    row = {"Representación": rep, "Modelo": "LinearSVC"}
    for metric, label in metrics.items():
        row[label] = scores[f"test_{metric}"].mean()
        row[f"{label}_std"] = scores[f"test_{metric}"].std()
    rows.append(row)

df_compare_optuna = pd.DataFrame(rows).sort_values("MCC", ascending=False).reset_index(drop=True)
df_compare_optuna.round(4)


### Resultados en test (LinearSVC + TF-IDF con hiperparámetros de Optuna)


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef)

# Test (se usa solo para la evaluación final)
test = pd.read_json("../data/interim/test_tok.jsonl", lines=True)
X_test = test["tokens"].map(flatten)
y_test = test["jailbreak"]

# Modelo con mejor CV tras Optuna: LinearSVC + TF-IDF
# (build_tuned_svc se define en la celda de "Resultados CV con Optuna")
with open("../data/processed/svc_tfidf_best_params.json") as f:
    best_params = json.load(f)

pipe = build_tuned_svc(build_vectorizer, best_params)
pipe.fit(X_train, y_train)          # el vectorizador se ajusta solo con train
y_pred = pipe.predict(X_test)

df_test_compare = pd.DataFrame([{
    "Representación": "TF-IDF + Optuna",
    "Modelo": "LinearSVC",
    "F1": f1_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred),
}])
df_test_compare.round(4)
